# 1 - E-commerce Product Recommendation - Recommendation Systems

![Recommendation](https://upload.wikimedia.org/wikipedia/commons/7/72/Shopping_cart.svg)

Bu çalışmada e-ticaret verileri üzerinden müşterilerin birlikte satın aldığı ürünleri kullanarak ürün öneri sistemi geliştireceğiz.

## Akış

1. Veriyi yükleme
2. Veriyi okuma ve inceleme
3. Veri temizleme
4. Feature engineering
5. User-product matrix oluşturma
6. Ürün benzerliklerini hesaplama
7. Ürün öneri fonksiyonu oluşturma
8. Sonuç

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

## 1. Veriyi Yükleme

In [ ]:
# Bu bölümde zip dosyasını Google Drive içinden açıp çalışma alanına çıkaracağım.

In [ ]:
zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/Online Retail Dataset.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content')

os.listdir('/content')[:20]

## 2. Veriyi Okuma ve İnceleme

In [ ]:
# Bu bölümde csv dosyasını okuyup veri setinin genel yapısını inceleyeceğim.

In [ ]:
file_path = '/content/online_retail.csv'

df = pd.read_csv(file_path, encoding='latin1')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## 3. Veri Temizleme

In [ ]:
# Bu bölümde recommendation sistemi için gereksiz veya sorunlu kayıtları temizleyeceğim.

In [ ]:
df = df.dropna(subset=['CustomerID', 'Description'])
df = df[df['Quantity'] > 0]
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

df.isnull().sum()

## 4. Feature Engineering

In [ ]:
# Bu bölümde veri tiplerini düzenleyip öneri sistemi için gerekli yardımcı sütunları oluşturacağım.

In [ ]:
df['CustomerID'] = df['CustomerID'].astype(int).astype(str)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

df.head()

## 5. User-Product Matrix Oluşturma

In [ ]:
# Bu bölümde müşteri-ürün etkileşim matrisi oluşturup satın alma bilgisini binary forma çevireceğim.

In [ ]:
user_product_matrix = df.groupby(['CustomerID', 'StockCode'])['Quantity'].sum().unstack().fillna(0)
user_product_matrix = user_product_matrix.apply(lambda col: (col > 0).astype(int))

user_product_matrix.iloc[:5, :5]

In [ ]:
user_product_matrix.shape

## 6. Ürün Benzerliklerini Hesaplama

In [ ]:
# Bu bölümde ürünler arasındaki benzerliği cosine similarity ile hesaplayacağım.

In [ ]:
product_matrix = user_product_matrix.T
product_similarity = cosine_similarity(product_matrix)

product_similarity_df = pd.DataFrame(
    product_similarity,
    index=product_matrix.index,
    columns=product_matrix.index
)

product_similarity_df.iloc[:5, :5]

## 7. Ürün Öneri Fonksiyonu Oluşturma

In [ ]:
# Bu bölümde seçilen bir ürün için en benzer ürünleri döndüren fonksiyonu oluşturacağım.

In [ ]:
product_names = (
    df[['StockCode', 'Description']]
    .drop_duplicates(subset='StockCode')
    .set_index('StockCode')['Description']
    .to_dict()
)

def recommend_products(stock_code, top_n=5):
    similar_scores = product_similarity_df[stock_code].sort_values(ascending=False)
    similar_scores = similar_scores.drop(stock_code, errors='ignore').head(top_n)
    recommendations = pd.DataFrame({
        'StockCode': similar_scores.index,
        'Similarity': similar_scores.values,
        'Description': [product_names.get(code, 'Unknown Product') for code in similar_scores.index]
    })
    return recommendations

In [ ]:
# Bu bölümde veri setindeki popüler ürünlerden biri için örnek öneriler alacağım.

In [ ]:
popular_product = df['StockCode'].value_counts().index[0]
popular_product, product_names.get(popular_product, 'Unknown Product')

In [ ]:
recommend_products(popular_product, top_n=5)

## 8. Sonuç

In [ ]:
# Bu bölümde proje sonucunu kısa bir özetle değerlendireceğim.

Bu projede e-ticaret ürünleri arasında müşteri satın alma davranışlarına göre öneri yapmak için item-based recommendation yöntemi kullanıldı. Elde edilen sonuçlara göre seçilen ürün için en benzer 5 ürün başarıyla önerildi.